# Exercise 69 — Mini-project: Log parser

## Problem

A web service produces lines like:

```
2026-01-15T10:00:01 INFO  request_id=abc12 user=alice path=/api/orders status=200 dur_ms=42
2026-01-15T10:00:02 WARN  request_id=abc13 user=bob   path=/api/orders status=200 dur_ms=812
2026-01-15T10:00:03 ERROR request_id=abc14 user=alice path=/api/login  status=500 dur_ms=15
```

The format is semi-structured: a fixed timestamp + level + a series of `key=value` pairs in any order. You need to turn this into a clean tabular CSV.

## Goals

1. **Parse** each line into a dict with the right types.
2. **Skip** lines that don't match the expected shape (real logs always have garbage in them).
3. **Write** the cleaned records to a CSV with a fixed schema.

## Setup

In [ ]:
from pathlib import Path

LOG = "web.log"
Path(LOG).write_text(
    "2026-01-15T10:00:01 INFO  request_id=abc12 user=alice path=/api/orders status=200 dur_ms=42\n"
    "2026-01-15T10:00:02 WARN  request_id=abc13 user=bob path=/api/orders status=200 dur_ms=812\n"
    "2026-01-15T10:00:03 ERROR request_id=abc14 user=alice path=/api/login status=500 dur_ms=15\n"
    "\n"                                                                  # blank line
    "this is not a log line at all\n"                                     # garbage
    "2026-01-15T10:00:04 INFO  request_id=abc15 user=carol path=/health status=200 dur_ms=3\n"
    "2026-01-15T10:00:05 INFO  partial line with no kvs\n"                # malformed
    "2026-01-15T10:00:06 INFO  request_id=abc16 user=alice path=/api/orders status=200 dur_ms=NOT_A_NUM\n",  # bad type
    encoding="utf-8",
)
print("setup ok")


## Task 1 — Parse one line

Write a function that parses a single log line into a dict. The expected shape:

```
{"ts": <str>, "level": <str>, "request_id": <str>, "user": <str>, "path": <str>, "status": <int>, "dur_ms": <int>}
```

Return `None` if the line doesn't have all required fields or if `status`/`dur_ms` aren't valid integers.

```python
def parse_line(line: str) -> dict | None:
    ...
```

Design hint:
- Strip the line. If empty → `None`.
- The first two whitespace-separated tokens are `ts` and `level`.
- The rest is a series of `key=value` pairs separated by whitespace. Split on `=` once per pair.
- Validate that all required keys are present and types parse.

Use `str.split` rather than regex for this format — it's faster and clearer.

In [ ]:
def parse_line(line: str) -> dict | None:
    pass  # TODO

good = parse_line("2026-01-15T10:00:01 INFO  request_id=abc12 user=alice path=/api/orders status=200 dur_ms=42")
assert good == {
    "ts": "2026-01-15T10:00:01",
    "level": "INFO",
    "request_id": "abc12",
    "user": "alice",
    "path": "/api/orders",
    "status": 200,
    "dur_ms": 42,
}

assert parse_line("") is None
assert parse_line("this is not a log line at all") is None
assert parse_line("2026-01-15T10:00:05 INFO  partial line with no kvs") is None
assert parse_line("2026-01-15T10:00:06 INFO  request_id=x user=a path=/p status=200 dur_ms=NOPE") is None
print("ok")


## Task 2 — Stream-parse a file

Write a generator that yields parsed records from a log file, skipping unparseable lines. Stream — do not load the whole file.

```python
def iter_parsed(path: str):
    ...
```

Verify: `list(iter_parsed(LOG))` has 4 valid records (the 3 INFO/WARN/ERROR ones plus carol's INFO), and the malformed/empty lines are dropped.

In [ ]:
def iter_parsed(path: str):
    pass  # TODO

records = list(iter_parsed(LOG))
assert len(records) == 4
assert records[0]["request_id"] == "abc12"
assert records[-1]["user"] == "carol"
print("ok")


## Task 3 — Write to CSV

Write a function that pipes the parsed stream into a CSV with a **fixed schema**: columns `[ts, level, request_id, user, path, status, dur_ms]` in that order.

```python
def parse_to_csv(log_path: str, csv_path: str) -> int:
    """Return rows written (excluding header)."""
```

Use `csv.DictWriter` with explicit `fieldnames=`. The records yielded by `iter_parsed` already match the schema.

In [ ]:
import csv
from pathlib import Path

def parse_to_csv(log_path: str, csv_path: str) -> int:
    pass  # TODO

n = parse_to_csv(LOG, "web_parsed.csv")
assert n == 4

text = Path("web_parsed.csv").read_text(encoding="utf-8")
assert text.splitlines()[0] == "ts,level,request_id,user,path,status,dur_ms"
assert text.splitlines()[1].startswith("2026-01-15T10:00:01,INFO,abc12,alice,/api/orders,200,42")
print("ok")
